# 05 · Predictive analytics: a forecast you can beat
The dashboard forecasts revenue with *trend + weekday pattern*. Here you test how good simple methods are by **backtesting**: hide the last 14 days, predict them, compare with what happened.

Metric: **MAPE** = average of |actual − forecast| / actual.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
import numpy as np

In [ ]:
daily = df.groupby("record_date")["revenue"].sum()
train, test = daily[:-14], daily[-14:]

def mape(actual, forecast):
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(np.abs(actual - forecast) / actual) * 100)

# Baseline 1: tomorrow = the last value we saw
naive = np.repeat(train.iloc[-1], len(test))
# Baseline 2: the average of the whole training period
mean_fc = np.repeat(train.mean(), len(test))
print(f"naive (last value): {mape(test, naive):.1f}% error")
print(f"training mean:      {mape(test, mean_fc):.1f}% error")

## Your turn
Build a **seasonal** forecast: for each test day, predict the average revenue of the **same weekday over the last 4 weeks of training**. Store the 14 predictions (a list or Series in test-day order) in `my_forecast`.

Hint: `train[-28:]` is the last 4 weeks; group it by `.index.dayofweek`.

In [ ]:
my_forecast = None

In [ ]:
if my_forecast is None:
    print("(write your forecast above first)")
else:
    my_error = mape(test, my_forecast)
    print(f"your seasonal forecast: {my_error:.1f}% error")
    check("beats the naive forecast", my_error < mape(test, naive), "weekday patterns repeat; averaging the same weekday should help")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.5))
train[-42:].plot(ax=ax, label="history", color="#64748b")
test.plot(ax=ax, label="actual (hidden)", color="#2563eb")
if my_forecast is not None:
    pd.Series(list(my_forecast), index=test.index).plot(ax=ax, label="your forecast", color="#d97706", linestyle="--")
ax.legend(); ax.set_ylabel("daily revenue"); ax.set_title("Backtest"); plt.show()

**Caveat:** this sample data was generated *with* a weekday pattern, so seasonal methods look great. On real data always compare against a naive baseline, and watch for outliers and one-off events.